In [2]:
import torch
import torch.nn as nn

In [3]:
d_model = 512
nhead = 8
src_vocab = 1000
tgt_vocab = 1000
batch_size = 2
src_len = 10
tgt_len = 8
PAD_IDX = 0
transformer = nn.Transformer(
    d_model=d_model,
    nhead=nhead,
    batch_first=True,
    activation=nn.ReLU(),
)

In [4]:
# 调用 transformer

# 词嵌入
src_embed = nn.Embedding(src_vocab, d_model, padding_idx=0)
tgt_embed = nn.Embedding(tgt_vocab, d_model, padding_idx=0)

# 模拟数据
src = torch.randint(0, src_vocab, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab, (batch_size, tgt_len))

# 手动把最后两个词设为 PAD
src[:, -2:] = PAD_IDX
tgt[:, -1:] = PAD_IDX


In [5]:
src_emb = src_embed(src)
tgt_emb = tgt_embed(tgt)

# 5. ⭐ 生成掩码
# 5.1 因果掩码（防止 Decoder 偷看未来）
tgt_mask = transformer.generate_square_subsequent_mask(tgt_len)

# 5.2 填充掩码（过滤掉 PAD 位置）
# 逻辑：找到序列中等于 PAD_IDX 的位置，作为 padding_mask
src_padding_mask = (src == PAD_IDX)  # shape: (batch_size, src_len)
tgt_padding_mask = (tgt == PAD_IDX)  # shape: (batch_size, tgt_len)
print(tgt_mask.shape)
print(src_padding_mask.shape)

torch.Size([8, 8])
torch.Size([2, 10])


In [6]:
output = transformer(
    src_emb,
    tgt_emb,
    tgt_mask=tgt_mask,
    src_key_padding_mask=src_padding_mask,
    tgt_key_padding_mask=tgt_padding_mask,
    memory_key_padding_mask=src_padding_mask,
)

output1 = transformer.encoder(src_emb)
print(output.shape, output1.shape)

torch.Size([2, 8, 512]) torch.Size([2, 10, 512])


D:\01_WorkSpace\02_Python\03_ai\nlp\.venv\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
